# Data prep for use in NER model training

In [1]:
# %pip install -r ../../Ner_Pipeline/requirements.txt
# %pip install -e ../../Ner_Pipeline

In [1]:
# Stops Jupyter getting messed up with edits
%load_ext autoreload
%autoreload 2

In [2]:
import ast
import pandas as pd
import json
from typing import List

# Screening to drop GeneProtein annotations (for now)

def parse_labels(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

def screen_annotations(labels: List, drop: str):
    return_labels = []
    for l in labels:
        if drop in l['labels']:
            continue
        else:
            return_labels.append(l)
    return return_labels

# Load LabelStudio TSV data
# df_raw = pd.read_csv("/Users/withers/Downloads/all_cleaned_labelstudio.tsv", sep="\t") # Cleaned up, all labels, baby
df_raw = pd.read_csv("/Users/withers/Downloads/var_alllab_xl.tsv", sep="\t") # No curation, all labels, xl
# df_raw = pd.read_csv("/Users/withers/Downloads/ray-variants-annotations.tsv", sep="\t")

df_raw['label'] = df_raw['label'].apply(parse_labels)
df_raw['label'] = df_raw['label'].apply(screen_annotations, drop="GeneProtein")
test = df_raw[df_raw['label'].apply(len) > 0]
what = test.iloc[0]['label']
w = what[0]
w['labels'][0] # OK

'RefSNP'

In [3]:
# See what label types are left
def list_labels(entry):
    labels = [lab for e in entry for lab in e["labels"]]
    return list(dict.fromkeys(labels))

all_labs = [lab for entry in test["label"].to_list() for lab in list_labels(entry)]
all_labs = list(dict.fromkeys(all_labs))
all_labs

['RefSNP', 'HGVSVar', 'Refgenome', 'StarAllele', 'ISCNVar']

In [4]:
# View what type 'Other' annotations are
def view_annotations(labels: List, view: str):
    return_labels = []
    for i, l in enumerate(labels):
        if view in l['labels']:
            return_labels.append((i, l))
        else:
            continue
    return return_labels

others = df_raw['label'].apply(view_annotations, view="Other")
others = others[others.apply(len) > 0]

print('These are missed and/or corrected annotations\n')
for r in others:
    for hit in r:
        print(hit[1]['text'])

These are missed and/or corrected annotations



In [5]:
from ner_pipeline.pipelines.data.preprocessing.article_normaliser import ArticleNormaliser, detect_section_headers, NERDatasetAnalyser

class VarParams:
    text_col = "text"
    label_col = "label"
    ent_label_key = "labels"

# Normaliser - use TSV parameters, strip headers, and enforce a 500 max character length
normaliser = ArticleNormaliser(
    params=VarParams(),
    section_header_func=detect_section_headers,
    max_len=500
)

print(f"Beginning Normalising {len(df_raw)} text snippets...")
df_normalised = normaliser.normalise(df_raw)

print(f"\nSplit into {len(df_normalised)} sentences.\n")
print("--- Results ---")
display(df_normalised[["sentence", "entities"]])

from ner_pipeline.pipelines.data.preprocessing.entity_processor import flatten_singleton_labels

# After normalisation, before IOB conversion:
df_normalised["entities"] = df_normalised["entities"].apply(
    lambda ents: flatten_singleton_labels(ents, ent_label_key="label")
)

## TODO - ERROR HERE, SKIPPING
# analyser = NERDatasetAnalyser(df_normalised, sent_col="sentence", ent_col="entities")
# stats = analyser.compute_entity_stats()
# print("\n--- DATASET HEALTH SUMMARY ---")
# print(f"Total Sentences Extracted: {stats.get('total_sentences')}")
# print(f"Total Labels Shifted & Retained: {stats.get('total_number_labels')}")
# pd.set_option('display.max_columns', None)
# print("\nLabel Distribution Count:")
# print(stats.get('labels_count'))


/Users/withers/GitProjects/OTAR3088/ner-model/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/withers/GitProjects/OTAR3088/ner-model/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This

Beginning Normalising 476 text snippets...


Sentencising Article--->: 100%|██████████| 476/476 [00:05<00:00, 93.40it/s] 


Split into 900 sentences.

--- Results ---


,sentence,entities
0,TagSNP identification for IL6R was performed...,[]
1,"Broad Institute of MIT/Harvard University, Ca...",[]
2,a total of four splice donor/acceptor variant...,[]
3,with a relatively high linkage disequilibrium...,[]
4,Additional file 1 )\nTagSNPs tagging <2 SNPs...,[]
...,...,...
895,MMP13 mediates collagen degradation and plays...,[]
896,"on the other hand, it reduces the secretion o...",[]
897,resulting in harmful phenotypes in chondrocyt...,"[{'start': 152, 'end': 162, 'label': ['HGVSVar..."
898,without obvious imaging abnormalities in the ...,[]


In [6]:
df_normalised.to_csv("variant_ner_xl_dataset.csv", index=False)

In [7]:
import ast
from ner_pipeline.pipelines.data.preprocessing.iob_converter import SpacyIOBConverter
from ner_pipeline.schemas.ner_dataset import IOBConfig, RawNerSchema, nlp

# df = pd.read_csv("variant_ner_dataset.csv")
df_normalised = df_normalised

def parse_labels(x):
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

df_normalised["entities"] = df_normalised["entities"].apply(parse_labels)
print(type(df_normalised["entities"].iloc[0]))

# Define column schema so the Converter knows where to look
schema = RawNerSchema(
    text_col="sentence",
    label_col="entities",
    ent_label_key="label"
)

# 4. Configure the Transformer IOB Settings
config = IOBConfig(
    schema=schema,
    tokenizer_backend=nlp,    # using en_core_sci_md as the exact backend tokenizer
    as_hf_dataset=True        # Exports straight to Hugging Face datasets.Dataset object
)

converter = SpacyIOBConverter(data=df_normalised, config=config)
hf_dataset = converter.convert()

print("\nHf dataset")
print(hf_dataset)

print("\n--- Row 0 ---")
print("Tokens:", hf_dataset[0]['tokens'][:10])
print("Tags:", hf_dataset[0]['tags'][:10])

output_file = "variant_iob_dataset.jsonl"
hf_dataset.to_json(output_file, orient="records", lines=True)

print(f"Saved dataset to {output_file}")

<class 'list'>


Map:   0%|          | 0/900 [00:00<?, ? examples/s]


Hf dataset
Dataset({
    features: ['tokens', 'tags'],
    num_rows: 900
})

--- Row 0 ---
Tokens: ['TagSNP', 'identification', 'for', ' ', 'IL6R', ' ', 'was', 'performed', 'using', 'the']
Tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved dataset to variant_iob_dataset.jsonl


In [8]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from datasets import DatasetDict

from ner_pipeline.pipelines.data.preprocessing.iob_converter import SpacyIOBConverter
from ner_pipeline.schemas.ner_dataset import IOBConfig, RawNerSchema, nlp

# df = pd.read_csv("variant_ner_dataset.csv")
df = df_normalised
df.head()

# --- NEW: GROUP-AWARE SPLITTING ON PMCID ---
# Split 1: 80% Train, 20% Temp
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
print(gss1)
train_idx, temp_idx = next(gss1.split(df, groups=df['pmcid']))

train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

# Split 2: Divide Temp exactly in half (10% Test, 10% Valid)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
test_idx, valid_idx = next(gss2.split(temp_df, groups=temp_df['pmcid']))

test_df = temp_df.iloc[test_idx].reset_index(drop=True)
valid_df = temp_df.iloc[valid_idx].reset_index(drop=True)

print(f"Unique papers -> Train: {train_df.pmcid.nunique()}, Test: {test_df.pmcid.nunique()}, Valid: {valid_df.pmcid.nunique()}")


# --- CONVERT EACH DISJOINT DATAFRAME INTO HUGGINGFACE IOB SEPARATELY ---
schema = RawNerSchema(text_col="sentence", label_col="entities", ent_label_key="label")
config = IOBConfig(schema=schema, tokenizer_backend=nlp, as_hf_dataset=True)

train_hf = SpacyIOBConverter(data=train_df, config=config).convert()
test_hf = SpacyIOBConverter(data=test_df, config=config).convert()
valid_hf = SpacyIOBConverter(data=valid_df, config=config).convert()

# Assemble into final layout
final_dataset_dict = DatasetDict({
    'train': train_hf,
    'test': test_hf,
    'validation': valid_hf
})

print(final_dataset_dict)

# Push the leak-proof dataset!
final_dataset_dict.push_to_hub("OTAR3088/Variant_AllLabel_Large")


GroupShuffleSplit(n_splits=1, random_state=42, test_size=0.2, train_size=None)
Unique papers -> Train: 22, Test: 3, Valid: 3


Map:   0%|          | 0/791 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 791
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 60
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 49
    })
})


...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########| 29.3kB / 29.3kB            

CommitInfo(commit_url='https://huggingface.co/datasets/OTAR3088/Variant_AllLabel_Large/commit/446d65ff0ebff9a84d8248eea18086cae7b83a6f', commit_message='Upload dataset', commit_description='', oid='446d65ff0ebff9a84d8248eea18086cae7b83a6f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/OTAR3088/Variant_AllLabel_Large', endpoint='https://huggingface.co', repo_type='dataset', repo_id='OTAR3088/Variant_AllLabel_Large'), pr_revision=None, pr_num=None)

### Scruffy data prep for universal 'variant' label

In [9]:
from datasets import load_dataset

ds = load_dataset("OTAR3088/Variant_AllLabel_Large")
dftr = ds['train'].to_pandas()
dftt = ds['test'].to_pandas()
dfvr = ds['validation'].to_pandas()

README.md:   0%|          | 0.00/522 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/22.3k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/29.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/791 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/60 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/49 [00:00<?, ? examples/s]

In [10]:
def swap_labels(tags: List, new_lab: str):
    cleaned = []
    for token in tags:
        if token == 'O':
            cleaned.append(token)
        else:
            new = token[:2]+new_lab
            cleaned.append(new)
    return cleaned

def relabel_annotations(df: pd.DataFrame, new_lab: str):
    for i, row in df.iterrows():
        if list(set(row['tags'])) != ['O']:
            new_row = swap_labels(tags=row['tags'], new_lab=new_lab)
            df.loc[i, 'tags'] = new_row
    return df


df_new = relabel_annotations(df=dftr, new_lab='Variant')

dftr_new = relabel_annotations(df=dftr, new_lab='Variant')
dftt_new = relabel_annotations(df=dftt, new_lab='Variant')
dfvr_new = relabel_annotations(df=dfvr, new_lab='Variant')

In [11]:
from datasets import Dataset, DatasetDict

# 1. Convert edited DataFrames back to HF Datasets
dataset = DatasetDict({
    "train":      Dataset.from_pandas(dftr_new, preserve_index=False),
    "test":       Dataset.from_pandas(dftt_new, preserve_index=False),
    "validation": Dataset.from_pandas(dfvr_new, preserve_index=False),
})

# 2. (Optional) Sanity check
print(dataset)
print(dataset["train"][0])

# 3. Push to the Hub
dataset.push_to_hub(
    "OTAR3088/Variant_UniLabel_Large"
)


DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 791
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 60
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 49
    })
})
{'tokens': ['TagSNP', 'identification', 'for', ' ', 'IL6R', ' ', 'was', 'performed', 'using', 'the', 'Tagger', 'method', 'implemented', 'in', 'Haploview', 'software', '(', 'v', '4.2', ';'], 'tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        : 100%|##########| 29.3kB / 29.3kB            

README.md:   0%|          | 0.00/522 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/OTAR3088/Variant_UniLabel_Large/commit/f5a44503988754c9d55876e24dc21eeef9a7501c', commit_message='Upload dataset', commit_description='', oid='f5a44503988754c9d55876e24dc21eeef9a7501c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/OTAR3088/Variant_UniLabel_Large', endpoint='https://huggingface.co', repo_type='dataset', repo_id='OTAR3088/Variant_UniLabel_Large'), pr_revision=None, pr_num=None)